In [1]:
import calendar
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.max_row',None)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 1, 12), datetime.date(2023, 1, 11))

In [2]:
# specify the number of business days
num_days = 1
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(num_days)
yesterday = today - num_business_days
#yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-12
yesterday: 2023-01-11 00:00:00


In [3]:
yesterday = yesterday.date()
a_year_ago = yesterday - timedelta(days=365)
yesterday, a_year_ago

(datetime.date(2023, 1, 11), datetime.date(2022, 1, 11))

### Get past one quarter data

In [4]:
sql = """
SELECT name
FROM buy 
ORDER BY name
"""
df = pd.read_sql(sql, const)

names = df['name'].values.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))
in_p

"'ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART'"

In [5]:
in_p = "'JMART'"
in_p

"'JMART'"

In [ ]:
one_qtr_ago = yesterday - timedelta(days=18)
one_qtr_ago

sql = """
SELECT * 
FROM price 
WHERE date >= '%s' AND name IN (%s) 
ORDER BY name, date"""
sql = sql % (one_qtr_ago, in_p)
print(sql)

In [ ]:
df = pd.read_sql(sql, const)
df

In [6]:
one_qtr_ago = yesterday - timedelta(days=18)
one_qtr_ago

sql = """
SELECT name, date, price 
FROM price 
WHERE date >= '%s' AND name IN (%s) 
ORDER BY name, date"""
sql = sql % (one_qtr_ago, in_p)
print(sql)


SELECT name, date, price 
FROM price 
WHERE date >= '2022-12-24' AND name IN ('JMART') 
ORDER BY name, date


In [7]:
df = pd.read_sql(sql, const)
df

,name,date,price
0,JMART,2022-12-26,39.50
1,JMART,2022-12-27,40.50
2,JMART,2022-12-28,41.00
3,JMART,2022-12-29,41.50
4,JMART,2022-12-30,40.75
5,JMART,2023-01-03,41.00
6,JMART,2023-01-04,40.75
7,JMART,2023-01-05,40.00
8,JMART,2023-01-06,40.25
9,JMART,2023-01-09,41.50


In [8]:
data_path = "../data/"
file_name = "Quarterly-Price-by-Name.csv"
output_file = data_path + file_name

df.set_index("name", inplace=True)
df.to_csv(output_file, header=None)

### Command Line Call Lwr-Dly & Upr-Dly Process

In [ ]:
name = 'SINGER'

In [ ]:
data_path = "../csv/"
file_name   = name + '-lower.csv'
input_file = data_path + file_name
col_names   = ['name','fm_date','to_date','days','fm_price','to_price','chg_amt','chg_pct','spd','avg','maxp','minp','qty']
df_lower = pd.read_csv(input_file, sep=',',  index_col=None, names=col_names)
df_lower.sort_values(by=['to_date'],ascending=[False])

In [ ]:
df_lower[df_lower['chg_pct'] <= -5].sort_values(by=['to_date'],ascending=[False])

In [ ]:
data_path = "../csv/"
file_name   = name + '-upper.csv'
input_file = data_path + file_name
col_names   = ['name','fm_date','to_date','days','fm_price','to_price','chg_amt','chg_pct','spd','avg','maxp','minp','qty']
df_upper = pd.read_csv(input_file, sep=',',  index_col=None, names=col_names)
df_upper.sort_values(by=['to_date'],ascending=[False])

In [ ]:
df_upper[df_upper['chg_pct'] >= 5].sort_values(by=['to_date'],ascending=[False])

### Concat

In [ ]:
frames = [df_lower, df_upper]
result = pd.concat(frames)
result.sort_values(by=['to_date'],ascending=[False])

### Sales table in port_lite

In [ ]:
sql = """
SELECT *
FROM sales
WHERE to_date = '%s'"""
sql = sql % today
print(sql)
df_sales = pd.read_sql(sql, conlite)
df_sales

In [ ]:
df_sales[df_sales['pct'] >= 5].sort_values(by=['pct'],ascending=[False])

In [ ]:
df_sales[df_sales['pct'] <= -5].sort_values(by=['pct'],ascending=[True])